# NumCompute Demo – End‑to‑End Pipeline

This notebook demonstrates:
- Loading a CSV file with `io.load_csv` (including missing values)
- Preprocessing: scaling, one‑hot encoding, imputation
- Statistics: mean, std/variance, histogram, quantiles
- Sorting / searching (top‑k, binary search)
- Ranking and percentiles
- Finite‑difference gradient approximation
- Performance benchmark (vectorised vs. Python loop)

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'numcompute').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from numcompute.io import load_csv
from numcompute.preprocessing import StandardScaler, OneHotEncoder, SimpleImputer
from numcompute.stats import mean,std,  histogram, quantile
from numcompute.sort_search import topk, binary_search, quickselect
from numcompute.rank import rank, percentile
from numcompute.optim import grad, jacobian
from numcompute.pipeline import FeatureUnion, Pipeline
from numcompute.metrics import accuracy, precision, recall, f1, confusion_matrix, mse, roc_curve, auc


In [ ]:
# %% [markdown]
# # 1. Load the CSV file

# %%
csv_path = str(PROJECT_ROOT / "demo" / "dataset" / "test_student_performance_data.csv") 
data, column_names = load_csv(csv_path, skip_header=True)
print(f"Shape of loaded data: {data.shape}")
print(f"Column names: {column_names}")
  


# Check for missing values
missing_counts = np.isnan(data).sum(axis=0)
print("Missing values per column:", missing_counts)

In [ ]:
# %% [markdown]
# # 2. Preprocessing Define numeric and categorical features (based on the dataset)

# %%
numeric_features = ['Age', 'StudyTimeWeekly', 'Absences', 'GPA']
categorical_features = [
    'Gender', 'Ethnicity', 'ParentalEducation', 'Tutoring',
    'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering'
]

# Get column indices
num_idx = [column_names.index(col) for col in numeric_features]
cat_idx = [column_names.index(col) for col in categorical_features]

X_num = data[:, num_idx]
X_cat = data[:, cat_idx]

# Impute numeric missing values (if any) with mean
imputer_num = SimpleImputer(strategy='mean')
X_num_imputed = imputer_num.fit_transform(X_num)

# Scale numeric features
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num_imputed)

# Impute categorical missing values with a constant (e.g., -1)
imputer_cat = SimpleImputer(strategy='constant', fill_value=-1)
X_cat_imputed = imputer_cat.fit_transform(X_cat)

# One‑hot encode categorical features
encoder = OneHotEncoder(handle_unknown='ignore')
X_cat_encoded = encoder.fit_transform(X_cat_imputed)

# Combine
X_processed = np.hstack([X_num_scaled, X_cat_encoded])
print(f"Preprocessed data shape: {X_processed.shape}")

In [ ]:
# %% [markdown]
# # 3. Statistics on GPA

# %%
gpa = X_num_imputed[:, 3]   # GPA column (last numeric)
print(f"Mean GPA: {mean(gpa):.4f}")
print(f"Std GPA : {std(gpa):.4f}")
print(f"25th percentile: {quantile(gpa, 0.25):.4f}")
print(f"Median (50th)   : {quantile(gpa, 0.5):.4f}")
print(f"75th percentile: {quantile(gpa, 0.75):.4f}")

counts, edges = histogram(gpa, n_bins=10)
print("Histogram counts (10 bins):", counts)

In [ ]:
# %% [markdown]
# # 4. Sorting and searching using real modules

# %%
print("Top 5 GPAs (values):", topk(gpa, 5, return_indices=False, sorted=True))

sorted_gpa = np.sort(gpa)
median_val = quantile(gpa, 0.5)
k = 499
kth_value = quickselect(gpa,k)
print(f"The {k+1}th smallest GPA is: {kth_value: .4f}")
idx, exists = binary_search(sorted_gpa, median_val)
print(f"Median GPA {median_val:.4f} found at sorted index: {idx} (exists: {exists})")

In [ ]:
# %% [markdown]
# # 5. Ranking and percentiles using real modules

# %%
sample = gpa[:10]
print("Sample GPA:", sample)
print("Ranks (average method):", rank(sample, method="average"))
print("90th percentile (linear):", percentile(gpa, 90, interpolation="linear"))


In [ ]:
# %% [markdown]
# # 6. Gradient estimation using real grad function

# %%
def func(x):
    return x[0]**2 + np.sin(x[1])

point = np.array([2.0, 0.5])
grad_val = grad(func, point, h=1e-6, method='central')
print("Gradient at point [2.0, 0.5]:", grad_val)
print("Analytical: [4.0, cos(0.5)] = [4.0, 0.87758]")
# Jacobian of a vector-valued function
def vector_func(x):
    return np.array([x[0]**2 + x[1], np.sin(x[0] * x[1])])

print('Estimated Jacobian:', jacobian(vector_func, point, h=1e-5), sep='\n')


In [ ]:
# 7. Reproducible loop versus vectorised benchmarks
import platform
from numcompute.benchmarking import run_all
print('Python:', platform.python_version(), 'NumPy:', np.__version__)
bench_results = run_all(n=2000, repeats=5, seed=42)


In [ ]:
# Define pipelines separately
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy='mean')),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy='constant', fill_value=-1)),
    ("encoder", OneHotEncoder(handle_unknown='ignore'))
])

# Wrap them using lambda-style transformers (simple approach)
class NumSelector:
    def transform(self, X):
        return X[:, num_idx]
    def fit(self, X, y=None):
        return self

class CatSelector:
    def transform(self, X):
        return X[:, cat_idx]
    def fit(self, X, y=None):
        return self

# Combine using FeatureUnion
union = FeatureUnion([
    ("num", Pipeline([
        ("selector", NumSelector()),
        ("num_pipe", num_pipeline)
    ])),
    ("cat", Pipeline([
        ("selector", CatSelector()),
        ("cat_pipe", cat_pipeline)
    ]))
])

X_union = union.fit_transform(data)

print("FeatureUnion output shape:", X_union.shape)

In [ ]:
# Simulated classification example
y_true = np.array([1, 0, 1, 1, 0, 1, 0])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1])

print("Accuracy :", accuracy(y_true, y_pred))
print("Precision:", precision(y_true, y_pred))
print("Recall   :", recall(y_true, y_pred))
print("F1 Score :", f1(y_true, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))

y_true_reg = np.array([3.0, 2.5, 4.0])
y_pred_reg = np.array([2.8, 2.7, 3.9])

print("MSE:", mse(y_true_reg, y_pred_reg))

# Simulated probability scores
y_true = np.array([0, 0, 1, 1])
y_scores = np.array([0.1, 0.4, 0.35, 0.8])

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

print("FPR:", fpr)
print("TPR:", tpr)
print("AUC:", roc_auc)

## Summary

This notebook demonstrated:
- Loading CSV with `io.load_csv`
- Preprocessing (imputation, scaling, one‑hot encoding) using `preprocessing`
- Statistics (`mean`, `std`, `histogram`, `quantile`) from `stats`
- Sorting, binary search, ranking, percentile using `sort_search` and `rank`
- Finite‑difference gradient estimation with `optim`
- Benchmarking vectorised NumPy vs. pure Python loops